# Classes and call flow: how al-molecular's code fits together

**Who this is for:** you know basic Python (variables, lists, dicts, loops, functions,
a bit of NumPy), for example from `python_from_zero.ipynb`, and you understand the
idea of active learning. Now you want to **read the real code**: what all the classes
are, why they exist, and **which function calls which** when an experiment runs.

**What you'll do:**

| Part | Python idea | al-molecular piece |
|---|---|---|
| 1 | classes, `__init__`, `self`, attributes | a surrogate that remembers what it learned |
| 2 | reading a real class | `SingleBackboneMVESurrogate` |
| 3 | interfaces, duck typing, abstract base classes | why any surrogate can plug into the loop; `molpal`'s `Model` |
| 4 | inheritance, overriding, `super()` | `SingleBackboneEmbeddingModel`, `ParallelMVEExplorer` |
| 5 | composition (objects inside objects) | Explorer → Model → Surrogate, the three layers |
| 6 | functions as values, factories, `**kwargs`, keyword-only args | `get_metric()`, `mve()` |
| 7 | `__call__`, `@property`, `hasattr`/`getattr` | `DictObjective`, `provides`, the `combine()` branch |
| 8 | **tracing real calls** | the full call tree of one AL round, for two surrogates |
| 9 | `argparse`, `main()`, `if __name__ == "__main__"` | how a command line becomes objects |
| 10 | saving state: `json`, `pickle`, resume | `iter_N/` checkpoints and `--resume` |
| 11 | putting it together | write your own surrogate class and run it inside the real loop |

Everything runs on a CPU, using **the repo's real classes** on a small fake pool
(3,000 molecules) so it's quick. Kernel: **py310**. Run the cells top to bottom.
Total runtime is a couple of minutes, mostly the first time PyTorch starts.

In [ ]:
import sys, inspect, json, pickle, tempfile
from pathlib import Path

import numpy as np

AL_REPO = Path("/N/slate/mengjing/repos/al-molecular")
sys.path.insert(0, str(AL_REPO))     # lets us `import run_experiment`, `import surrogates`, `import molpal`
sys.path.insert(0, ".")              # lets us import calltrace.py from this folder

import run_experiment as rx          # the repo's driver script, imported as a module
import surrogates                    # all the surrogate classes
from calltrace import trace          # a small helper that prints "who called whom"

import torch
torch.manual_seed(0)                 # main() does the same with --seed, so neural-net results repeat

WORK = Path(tempfile.mkdtemp(prefix="al_classes_nb_"))   # every run folder this notebook makes goes in here
def new_run_dir():
    return Path(tempfile.mkdtemp(dir=WORK))

print("imported run_experiment from", rx.__file__)
print("scratch folder for runs:", WORK)

Importing `run_experiment` does **not** start an experiment. It only defines functions
and classes. The experiment starts only when `main()` is called, and Part 9 shows why.

Our fake pool: 3,000 molecules, two fake "backbones" (embedding tables), and an oracle
`{name: docking score}` where lower = better. These have exactly the shapes the real
code expects.

In [ ]:
rng = np.random.default_rng(0)
N = 3000
pool_smiles = [f"mol_{i}" for i in range(N)]             # stand-ins for SMILES strings
hidden = rng.normal(size=(N, 4)).astype("float32")        # the "true chemistry"
emb_dict = {                                              # {backbone name: (N, D) array}, like load_embeddings()
    "molformer": np.c_[hidden, rng.normal(size=(N, 12))].astype("float32"),   # 16 columns
    "grover":    (hidden @ rng.normal(size=(4, 8))).astype("float32"),        #  8 columns
}
true_scores = -8 - (hidden[:, 0] + 0.5 * hidden[:, 1] + 0.3 * hidden[:, 2] * hidden[:, 3])
oracle = dict(zip(pool_smiles, true_scores.astype(float)))  # {smiles: score}, like load_oracle()

print({k: v.shape for k, v in emb_dict.items()}, "| oracle size:", len(oracle))

---
## Part 1: Why classes?

A surrogate has to **remember** things between two calls. `fit` learns something, and
later `predict` needs it. With plain functions you'd have to carry what was learned
around yourself:

```python
weights = fit(X, y)                 # you hold on to 'weights'...
mu = predict(weights, X_new)        # ...and hand it back every time
```

A **class** bundles the data (the *state*) with the functions that use it (the
*methods*). Here's the smallest possible surrogate: it predicts the average score it
was trained on.

In [ ]:
class MeanSurrogate:
    def __init__(self):              # runs once, when you create the object
        self.mean = None             # an ATTRIBUTE: a variable stored on the object

    def fit(self, X, y):             # a METHOD: a function that belongs to the class
        self.mean = float(np.mean(y))

    def predict(self, X):
        return np.full(len(X), self.mean)

s = MeanSurrogate()                  # create an object (an "instance") -> calls __init__
print("before fit:", s.mean)
s.fit(emb_dict["molformer"][:10], np.arange(10))
print("after fit: ", s.mean)
print("predict:   ", s.predict(emb_dict["molformer"][:3]))

**What is `self`?** It's the object the method was called on. When you write
`s.fit(X, y)`, Python actually runs `MeanSurrogate.fit(s, X, y)`, so `self` *is* `s`.
That's how `fit` can store `self.mean` and `predict` can read it later.

Each object has its **own** attributes. Two surrogates don't share what they learned:

In [ ]:
a, b = MeanSurrogate(), MeanSurrogate()
a.fit(None, [1, 2, 3])
b.fit(None, [100, 200])
print("a.mean =", a.mean, "  b.mean =", b.mean)
print("same thing, the long way:", MeanSurrogate.predict(a, [0, 0]))
print("all of a's attributes:", vars(a))

That last point matters in the real code. `MVEExplorer` keeps **one** surrogate object
for the whole campaign. For `SingleBackboneMVESurrogate`, the neural-net weights stay on
the object from round to round, so each round continues training from where the last
one stopped ("warm start"). The comment `# Warm-start from previous round's weights` in
`surrogates.py` refers to exactly this. Some surrogates choose the opposite on purpose:
`LTAllSurrogate` builds fresh networks every round. Same object, different decision
about what state to keep.

---
## Part 2: Reading a real class

Here's a more realistic toy, built to mirror the real `SingleBackboneMVESurrogate`:

* it **normalizes** `y` before training (subtract the mean, divide by the std) and
  remembers those two numbers as `self._ym` and `self._ys`
* `predict` returns **two** arrays, `(mu, sigma)`, the guess and the uncertainty
* sigma comes from 5 slightly different linear models (a small *ensemble*) disagreeing

A leading underscore (`_ym`) is a naming convention meaning "internal, don't touch from
outside". Python doesn't enforce it.

In [ ]:
class LinearEnsembleSurrogate:
    def __init__(self, n_models=5, seed=0):
        self.n_models = n_models
        self._rng = np.random.default_rng(seed)
        self._ym = self._ys = None
        self._ws = []

    def fit(self, X, y, epochs=None, batch=None):    # accepts (and ignores) epochs/batch, like the real ones
        self._ym = float(y.mean())
        self._ys = float(y.std()) + 1e-8
        y_norm = (y - self._ym) / self._ys
        X1 = np.c_[X, np.ones(len(X))]
        self._ws = []
        for _ in range(self.n_models):
            rows = self._rng.integers(0, len(X), len(X))           # bootstrap resample
            self._ws.append(np.linalg.lstsq(X1[rows], y_norm[rows], rcond=None)[0])

    def predict(self, X):
        X1 = np.c_[X, np.ones(len(X))]
        preds = np.stack([X1 @ w for w in self._ws])               # (n_models, n)
        mu_n, sig_n = preds.mean(0), preds.std(0)
        return mu_n * self._ys + self._ym, sig_n * self._ys        # back to original units

    def __repr__(self):                                            # what print() shows
        return f"LinearEnsembleSurrogate(n_models={self.n_models}, fitted={bool(self._ws)})"

lin = LinearEnsembleSurrogate()
print(lin)
X, y = emb_dict["molformer"][:200], -true_scores[:200]
lin.fit(X, y)
print(lin)
mu, sigma = lin.predict(emb_dict["molformer"][200:205])
print("mu   ", mu.round(2), "\nsigma", sigma.round(3))

Now read the **real** class. `inspect.getsource` prints any function's or class's code.
Compare it with ours: same `__init__` / `fit` / `predict` shape and the same `_ym`/`_ys`
normalization. The only big difference is that the "model" is a small PyTorch network
(`_DualMVEModel`) trained by `_train_model`.

In [ ]:
print(inspect.getsource(surrogates.SingleBackboneMVESurrogate))

Things to notice in the real code:

* `__init__` takes **parameters with defaults** (`spearman_weight: float = 0.1`). The
  `: float` parts are *type hints*. They document the code and Python doesn't enforce them.
* `fit` builds nothing new. It reuses `self._model` from `__init__`, which is the warm start.
* `predict` "de-normalises the mean only": `mu_n * self._ys + self._ym`. It does **not**
  rescale sigma. (Ours does. This is a real difference worth knowing when you read sigma
  values from this surrogate.)
* `-> tuple[np.ndarray, np.ndarray]` is a type hint for the return value: "returns two arrays".

---
## Part 3: Interfaces, so any surrogate can plug in

The AL loop never asks *what kind* of surrogate it has. It just calls `.fit(...)` and
`.predict(...)`. Any object with those two methods works. This is called **duck typing**
("if it walks like a duck and quacks like a duck...").

The docstring at the top of `surrogates.py` states the contract:

```
All surrogates share the same fit / predict interface:
  fit(X: np.ndarray, y: np.ndarray, epochs: int)
  predict(X: np.ndarray) -> (mu: np.ndarray, sigma: np.ndarray)
```

Here's a tiny AL loop that only relies on that contract. We give it two very different
surrogates.

In [ ]:
def tiny_loop(surrogate, X_all, y_all, init=60, batch=60, rounds=4, seed=1):
    r = np.random.default_rng(seed)
    labeled = np.zeros(len(X_all), bool)
    labeled[r.choice(len(X_all), init, replace=False)] = True
    top = set(np.argsort(y_all)[::-1][:50].tolist())                 # y is sign-flipped: higher = better
    for _ in range(rounds):
        surrogate.fit(X_all[labeled], y_all[labeled])                 # <- the contract
        cand = np.where(~labeled)[0]
        mu, sigma = surrogate.predict(X_all[cand])                    # <- the contract
        labeled[cand[np.argsort(mu)[::-1][:batch]]] = True
    return len(top & set(np.where(labeled)[0].tolist())) / 50

class MeanSurrogate2(MeanSurrogate):                                  # (Part 4 explains this line)
    def predict(self, X):
        return super().predict(X), np.zeros(len(X))                   # now returns (mu, sigma)

X_all, y_all = emb_dict["molformer"], -true_scores
for s in [MeanSurrogate2(), LinearEnsembleSurrogate()]:
    print(f"{type(s).__name__:26s} top-50 recall: {tiny_loop(s, X_all, y_all):.0%}")

The mean surrogate guesses the same thing for every molecule, so it is no better than
random. The linear one actually learns. **The loop didn't change at all.** That's the
point of an interface.

What happens when an object *breaks* the contract? Python only finds out when the
missing method is called, partway through the run. `try` / `except` catches the error
so this notebook keeps running:

In [ ]:
class Broken:
    def fit(self, X, y):
        pass
    # forgot predict!

try:
    tiny_loop(Broken(), X_all, y_all)
except AttributeError as err:
    print("AttributeError:", err)

To catch that mistake **earlier**, `molpal` uses an **abstract base class** (ABC). The
class `Model` in `molpal/models/base.py` lists methods marked `@abstractmethod`. Python
refuses to create an object of any class that hasn't written all of them:

In [ ]:
from molpal.models.base import Model

try:
    Model(test_batch_size=10)
except TypeError as err:
    print("TypeError:", err)

So `Model` is a **template**, not something you use directly. Every real model, MPN or
the embedding wrapper (`EmbeddingMVEModel`), must fill in `train`, `get_means`,
`get_means_and_vars`, `provides` and so on. Here's the same idea on a small scale:

In [ ]:
from abc import ABC, abstractmethod

class Surrogate(ABC):
    @abstractmethod
    def fit(self, X, y): ...
    @abstractmethod
    def predict(self, X): ...

class OnlyFit(Surrogate):
    def fit(self, X, y):
        pass

try:
    OnlyFit()
except TypeError as err:
    print("caught at creation time ->", err)

Note that `surrogates.py` itself does **not** use an ABC. Its surrogates follow the
contract by convention (duck typing). The ABC sits one layer up, on `Model`. Part 5
explains the layers.

---
## Part 4: Inheritance, reusing a class and changing one piece

`class Child(Parent):` means "a Child **is a** Parent, except where I say otherwise".
The child gets all the parent's methods for free, and can **override** (replace) any of
them. `super().method(...)` calls the parent's version from inside the child.

You already saw it: `MeanSurrogate2(MeanSurrogate)` reused `__init__` and `fit` and
changed only `predict`, which called `super().predict(X)` to reuse the parent's logic.

The repo uses inheritance in a few key places. First, the model wrapper:

In [ ]:
from molpal.models.mvemodels import EmbeddingMVEModel, SingleBackboneEmbeddingModel

# __mro__ = "method resolution order": where Python looks for a method, in order
print(" -> ".join(k.__name__ for k in SingleBackboneEmbeddingModel.__mro__))

own = [n for n in vars(SingleBackboneEmbeddingModel)
       if (not n.startswith("_") or n == "__init__")]       # skip Python's own bookkeeping entries
print("defined in SingleBackboneEmbeddingModel itself:", own)
print("inherited from EmbeddingMVEModel, e.g.:", [n for n in ("train", "_get_X", "get_means_and_vars") if n not in own])

`SingleBackboneEmbeddingModel` only overrides `__init__`. That `__init__` calls
`super().__init__(...)` to do all the normal setup, then keeps just one backbone:

In [ ]:
print(inspect.getsource(SingleBackboneEmbeddingModel.__init__))

The bigger example is the explorers. `ParallelMVEExplorer` runs the *same* AL loop, but
spreads pool prediction across many GPU jobs (needed for the 100-million-molecule
pools). Let's see which methods it replaces and which it reuses unchanged:

In [ ]:
def where_defined(cls, names):
    for n in names:
        for k in cls.__mro__:
            if n in vars(k):
                print(f"  {cls.__name__}.{n:12s} comes from {k.__name__}")
                break

where_defined(rx.ParallelMVEExplorer, ["__init__", "run", "_checkpoint", "_best", "_recall", "_save_final"])

It **overrides** `__init__`, `run` and `_checkpoint`, and **inherits** the small helpers
`_best`, `_recall` and `_save_final`. There's a subtle lesson in its `__init__`: it sets
some things **before** calling `super().__init__()`, because the parent's `__init__`
already *uses* them (to draw the random starting molecules). The order of lines in an
`__init__` can matter:

In [ ]:
src = [line for line in inspect.getsource(rx.ParallelMVEExplorer.__init__).splitlines()
       if not line.strip().startswith("#")]                   # drop comment lines
end = next(i for i, line in enumerate(src) if "super().__init__" in line)
print("\n".join(src[:end + 3]))
print("        ...")

A third subclass lives in `adaptive_acq/explorer.py`: `AdaptiveMVEExplorer(exp.MVEExplorer)`.
It is the follow-up project's version of the loop, with a β controller, a prefilter and
so on. It also starts with `super().__init__(acq="greedy", **kwargs)`.

---
## Part 5: Composition, the three layers

Inheritance is "**is a**". Composition is "**has a**": an object stores other objects as
attributes. al-molecular is built from three layers, each holding the next:

```
MVEExplorer                      the AL loop: picks molecules, keeps the labeled set, writes results
  │  .model  ─────────────┐
  ▼                       │      speaks SMILES strings
EmbeddingMVEModel         │      the wrapper (a molpal Model): turns SMILES -> embedding rows,
  │  .surrogate ──────┐   │      and sigma -> variance
  ▼                   │   │
SingleBackboneMVESurrogate (or EnsembleFusion, LTAll, ...)   speaks NumPy arrays: fit(X, y), predict(X)
```

**Why the middle layer?** The explorer thinks in molecules (SMILES strings). The
surrogate thinks in number tables. The wrapper translates between them: it keeps
`smi2idx` (SMILES → row number) and `_get_X(smiles_list)` (→ the rows of the embedding
table). It also satisfies `molpal`'s `Model` interface, so the same explorer code works
for MPN too.

Let's build a **real** `MVEExplorer` on our fake pool and look inside.

In [ ]:
explorer = rx.MVEExplorer(
    emb_dict=emb_dict, pool_smiles=pool_smiles, oracle=oracle,
    surrogate_type="single", backbone="molformer", acq="greedy",
    init_size=100, batch_size=100, n_rounds=3, topk=50,
    run_dir=new_run_dir(), seed=42,
    surrogate_epochs=10,               # the real default is 50; 10 keeps this fast
)

In [ ]:
print("explorer           :", type(explorer).__name__)
print("explorer.model     :", type(explorer.model).__name__)
print("explorer.model.surrogate:", type(explorer.model.surrogate).__name__)
print("explorer.acq_fn    :", explorer.acq_fn.__name__)
print()
first_two = {str(k): round(float(v), 2) for k, v in list(explorer.labeled_scores.items())[:2]}
print("labeled so far     :", len(explorer.labeled_idx), "molecules, e.g.", first_two)
print("smi2idx['mol_5']   :", explorer.model.smi2idx["mol_5"])
print("_get_X(['mol_0','mol_1']).shape:", explorer.model._get_X(["mol_0", "mol_1"]).shape, " (16 molformer columns)")

**Your turn:** the wrapper converts sigma to *variance* on the way out. Find the line by
running the next cell. Why does it do that? (Hint: look at what `ucb()` in
`molpal/acquirer/metrics.py` does with its second argument.)

In [ ]:
print(inspect.getsource(EmbeddingMVEModel.get_means_and_vars))

---
## Part 6: Functions as values, factories, and `**kwargs`

In Python a function is a value like any other: you can put it in a dictionary, return
it from another function, or store it on an object. `explorer.acq_fn` above *is* the
`greedy` function. It was chosen by name:

In [ ]:
from molpal.acquirer.metrics import get_metric

f = get_metric("ucb")
print(f, "->", f(np.array([1.0, 2.0]), np.array([0.25, 0.0])))   # mu + 2*sqrt(var)
print(inspect.getsource(get_metric))

That's a **dictionary of functions**, plus a `KeyError` turned into a friendlier
`ValueError`. The same "pick by name" trick builds the surrogate. `mve()` in
`molpal/models/__init__.py` is a **factory**: you give it a string like `"ensemble"` and
it builds the right surrogate class and wraps it in the right model class. Here's its
heart (abridged):

In [ ]:
src = inspect.getsource(sys.modules["molpal.models"].mve)
body = src[src.index("dims = {"):src.index('if surrogate_type == "learned"')]
print(body)

**`**kwargs`** ("keyword arguments") collects any extra `name=value` arguments into a
dict, so a function can pass them along without listing them all. `mve(..., **kwargs)`
forwards extras to the model class, and `surrogate_kwargs` is a dict that gets
*unpacked* with `**` into the surrogate's `__init__`:

In [ ]:
def build(kind, **kwargs):
    print("  build got kwargs =", kwargs)
    return dict(kind=kind, **kwargs)              # ** unpacks the dict back into name=value pairs

print(build("ltall"))
print(build("ltall", top_weight=1.0, M=5))
settings = {"lr": 3e-4, "epochs": 200}
print(build("ltall", **settings))                 # same as build("ltall", lr=3e-4, epochs=200)

**Keyword-only arguments.** In `EmbeddingMVEModel.train(self, xs, ys, *, featurizer=None,
retrain=False, epochs=None, batch=None, **kwargs)`, the bare `*` means "everything after
this **must** be passed by name". That prevents silent mix-ups like passing epochs where
batch was meant:

In [ ]:
def train(xs, ys, *, epochs=None, batch=None):
    return f"epochs={epochs}, batch={batch}"

print(train([1], [2], epochs=5))
try:
    train([1], [2], 5)              # positional -> refused
except TypeError as err:
    print("TypeError:", err)

---
## Part 7: Special methods, properties, and asking an object what it can do

**`__call__`** lets an object be called like a function. `run_experiment.py` has a tiny
example, `DictObjective`: an "objective" you call with a list of SMILES to get scores
back. (The sign flip `c = -1` means "we minimize the docking score".)

In [ ]:
print(inspect.getsource(rx.DictObjective))
obj = rx.DictObjective(oracle, minimize=True)
result = obj(["mol_0", "mol_1", "not_a_molecule"])    # calling the OBJECT runs __call__
print({k: None if v is None else round(float(v), 2) for k, v in result.items()})

`molpal`'s `Model` has one too: `model(xs)` runs `model.apply(xs)`.

**`@property`** makes a method look like an attribute (no parentheses). The wrapper
uses it to *declare* what it provides, and the rank-fusion surrogate uses it for
`n_components`:

In [ ]:
print("provides:", explorer.model.provides, "  type_:", explorer.model.type_)     # no () !
fusion = surrogates.EnsembleFusionSurrogate(dims={"molformer": 16, "grover": 8})
print("EnsembleFusionSurrogate.n_components:", fusion.n_components)

**Asking an object what it can do: `hasattr` / `getattr`.** This is the most important
idea for understanding the call flow. Two places in the real code take a *different
path* depending on what the surrogate has:

1. `EmbeddingMVEModel.train`: `if getattr(self.surrogate, "needs_smiles", False):` passes
   SMILES too (the fine-tuning surrogates need them).
2. `_chunked_get_means_and_vars`: `if hasattr(surrogate, "combine"):` uses the rank-fusion
   path. It predicts per chunk, then ranks **once** over the whole pool. (Ranking per
   chunk was one of the three fusion bugs fixed on 2026-10-10.)

In [ ]:
for cls in [surrogates.SingleBackboneMVESurrogate, surrogates.EnsembleFusionSurrogate,
            surrogates.LTAllSurrogate, surrogates.SingleBackboneFinetuneScheduleSurrogate]:
    print(f"{cls.__name__:42s} has combine: {hasattr(cls, 'combine')!s:5s}  "
          f"needs_smiles: {getattr(cls, 'needs_smiles', False)}")

In [ ]:
src = inspect.getsource(rx._chunked_get_means_and_vars)
print(src[src.index("    surrogate = getattr"):])          # skip the long docstring

---
## Part 8: Tracing the real call flow

Now we watch the code run. `trace(...)` runs something and prints **every al-molecular
function it called**, indented by who-called-whom, with the file and line where each is
defined. Inherited methods show the class that *defines* them, e.g.
`EmbeddingMVEModel.train (on a SingleBackboneEmbeddingModel)`. Repeats are collapsed.

**8a. Building an explorer.** What happens inside `MVEExplorer(...)`?

In [ ]:
explorer = trace(lambda: rx.MVEExplorer(
    emb_dict=emb_dict, pool_smiles=pool_smiles, oracle=oracle,
    surrogate_type="single", backbone="molformer", acq="greedy",
    init_size=100, batch_size=100, n_rounds=1, topk=50,
    run_dir=new_run_dir(), seed=42, surrogate_epochs=3,
))

Read it top to bottom:

* `mve` is the factory from Part 6. It builds the surrogate (whose `__init__` builds the
  PyTorch network: one backbone net plus two MVE heads) and then the wrapper. The
  wrapper's `__init__` chains up through `super().__init__` twice:
  `SingleBackboneEmbeddingModel` → `EmbeddingMVEModel` → `Model`.
* `get_metric` picks the acquisition function by name.
* `true_top_k_set` computes the answer key, used only for grading.
* `_best` belongs to the `[init] ... best=` printout.

**8b. One round of active learning.**

In [ ]:
history = trace(explorer.run)

That tree **is** the AL loop. Here it is with the jobs labeled:

| Call | Job |
|---|---|
| `EmbeddingMVEModel.train` → `_get_X` → `SingleBackboneMVESurrogate.fit` → `_train_model` | **fit**: SMILES → rows → train the network. The repeated block is one training step (`forward` + loss) per mini-batch per epoch |
| `_chunked_get_means_and_vars` → `get_means_and_vars` → `_get_X` → `predict` → `_predict_model` | **predict** every undocked molecule, in chunks |
| `greedy` | **acquire**: turn `(mu, var)` into scores |
| (no function call: `np.argsort(...)`, then the oracle lookup) | **pick and "dock"** the batch |
| `_recall`, `_best` | grade and print |
| `_checkpoint`, `_save_final` | write `iter_1/` and `history.json` (Part 10) |

**8c. Same loop, different surrogate.** Now `surrogate_type="ensemble"` and `acq="ucb"`.
Watch the predict step take the `hasattr(surrogate, "combine")` branch from Part 7.
(`max_depth=3` hides the PyTorch-level detail.)

In [ ]:
explorer_f = rx.MVEExplorer(
    emb_dict=emb_dict, pool_smiles=pool_smiles, oracle=oracle,
    surrogate_type="ensemble", backbone="molformer", acq="ucb",
    init_size=100, batch_size=100, n_rounds=1, topk=50,
    run_dir=new_run_dir(), seed=42, surrogate_epochs=3,
)
_ = trace(explorer_f.run, max_depth=3)

Differences from 8b:

* **fit**: `EnsembleFusionSurrogate.fit` splits the wide input back into one block per
  backbone (`_split`) and trains one `SingleBackboneMVESurrogate` **per backbone** (`x2`).
  That's composition again: a surrogate made of surrogates.
* **predict**: `_chunked_predict_components` → `predict_components` (per-backbone means,
  per chunk), then **one** `combine` over the whole pool, which calls `_desc_ranks` once
  per backbone.
* **acquire**: `ucb` instead of `greedy`, because `acq="ucb"`.

**Same `MVEExplorer.run` code, different call tree.** The surrogate object decides.

---
## Part 9: From the command line to objects, `main()`

A real experiment starts from a shell command, for example:

```bash
python run_experiment.py --mode mve --surrogate ensemble --backbones grover molformer unimol \
    --acq greedy --init-size 2104 --batch-size 2104 --n-rounds 5 --topk 1000 --run-dir runs/x
```

At the bottom of `run_experiment.py`:

```python
if __name__ == "__main__":
    main()
```

`__name__` is `"__main__"` only when the file is **run** as a script. When it's
**imported** (as we did), `__name__` is `"run_experiment"`, so `main()` doesn't run.
That's why our `import` was safe.

In [ ]:
print("when imported, rx.__name__ =", repr(rx.__name__))

`main()` begins with `parse_args()`, which uses **argparse** to turn the command-line
words (`sys.argv`) into an object with one attribute per option. We can call it with
fake words to see the result. Unlisted options get their defaults.

In [ ]:
saved = sys.argv
sys.argv = ["run_experiment.py", "--mode", "mve", "--surrogate", "ensemble",
            "--backbones", "grover", "molformer", "unimol", "--acq", "ucb",
            "--init-size", "2104", "--batch-size", "2104"]
try:
    args = rx.parse_args()
finally:
    sys.argv = saved                    # always put it back

print(type(args).__name__)
for k in ["mode", "surrogate", "backbones", "acq", "init_size", "batch_size", "n_rounds", "seed", "parallel_predict"]:
    print(f"  args.{k:17s} = {getattr(args, k)!r}")

Note the dashes become underscores: `--init-size` → `args.init_size`.

Then `main()` builds the objects and starts the loop. We won't run it here (it loads the
real 2-million-molecule embeddings), but this is its call structure for `--mode mve`,
read from the code:

```
main()
├── parse_args()                                      command line -> args
├── load_oracle()                                     data/...scores.csv.gz -> {smiles: score}
├── load_embeddings(backbones)                        -> EmbeddingFeaturizer(...).load() -> (emb_dict, pool_smiles)
├── keep only molecules that have a score             (the non-parallel branch)
├── explorer = MVEExplorer(...)                       = Part 8a   (or ParallelMVEExplorer if --parallel-predict)
├── history = explorer.run()                          = Part 8b, repeated n_rounds times
└── print the results table
```

With `--mode molpal` (the MPN baseline), the middle part builds a `MolPALExplorer`
instead. It's a separate class with its own loop, not a subclass, because MPN reads SMILES
directly and needs no embeddings.

**Your turn:** open `run_experiment.py` at `def main` (around line 1585) and find each
line of that tree.

---
## Part 10: State on disk, checkpoints and resume

Long runs crash (out of memory, time limits, ...), so every round
`MVEExplorer._checkpoint` writes what's needed to continue: `iter_N/state.json` (a
small human-readable summary) and `iter_N/scores.pkl` (a **pickle**: Python's way of
saving any object, here the `{smiles: score}` dict of everything docked so far).

Run 3 rounds:

In [ ]:
run_dir = new_run_dir()
ex3 = rx.MVEExplorer(
    emb_dict=emb_dict, pool_smiles=pool_smiles, oracle=oracle,
    surrogate_type="single", backbone="molformer", acq="greedy",
    init_size=100, batch_size=100, n_rounds=3, topk=50,
    run_dir=run_dir, seed=42, surrogate_epochs=10,
)
_ = ex3.run()

In [ ]:
for p in sorted(run_dir.rglob("*")):
    print(p.relative_to(run_dir))
print("\niter_2/state.json:", (run_dir / "iter_2" / "state.json").read_text())

with open(run_dir / "iter_2" / "scores.pkl", "rb") as f:     # "rb" = read bytes
    scores_r2 = pickle.load(f)
print(type(scores_r2).__name__, "with", len(scores_r2), "molecules")

`find_resume_checkpoint` finds the newest **complete** round (both files present). With
`--resume`, `main()` passes that into the explorer's `__init__`, which then takes its
*other* branch: rebuild the labeled set from the saved scores instead of drawing a random
start. Let's pretend the run had been launched with `n_rounds=4` and crashed after round 3:

In [ ]:
resume_round, resume_scores = rx.find_resume_checkpoint(run_dir)
print("newest complete round:", resume_round, "| molecules saved:", len(resume_scores))

ex4 = rx.MVEExplorer(
    emb_dict=emb_dict, pool_smiles=pool_smiles, oracle=oracle,
    surrogate_type="single", backbone="molformer", acq="greedy",
    init_size=100, batch_size=100, n_rounds=4, topk=50,
    run_dir=run_dir, seed=42, surrogate_epochs=10,
    resume_scores=resume_scores, resume_round=resume_round,
)
hist = ex4.run()
print("\nhistory has rounds:", [h["round"] for h in hist])

It ran only round 4, and the history still covers rounds 1–4 (rounds 1–3 were read back
from `state.json`). One subtlety: the *network weights* are not saved, only the scores,
so after a resume the surrogate starts fresh. For warm-started surrogates that means a
resumed run is not bit-identical to an uninterrupted one.

---
## Part 11: Your own surrogate inside the real loop

Composition pays off here. Because the explorer only talks to `self.model`, and the model
only talks to `self.surrogate` through `fit`/`predict`, we can **swap in our own
surrogate object** and the real `MVEExplorer.run` will use it without knowing.

We'll put `LinearEnsembleSurrogate` from Part 2 inside a real explorer:

In [ ]:
ex_mine = rx.MVEExplorer(
    emb_dict=emb_dict, pool_smiles=pool_smiles, oracle=oracle,
    surrogate_type="single", backbone="molformer", acq="ucb",
    init_size=100, batch_size=100, n_rounds=1, topk=50,
    run_dir=new_run_dir(), seed=42,
)
ex_mine.model.surrogate = LinearEnsembleSurrogate()     # <- the swap
_ = trace(ex_mine.run)

The call tree is the same as in 8b, but where `SingleBackboneMVESurrogate.fit` used to
be there's now **nothing**: our class lives in this notebook, not in al-molecular, so the
tracer doesn't list it. The explorer, the wrapper and `ucb` all ran unchanged around it.

Now a fair comparison: 4 rounds, the real neural surrogate vs ours, same seed.

In [ ]:
import contextlib, io

def run_with(make_surrogate=None, rounds=4, seed=42):
    with contextlib.redirect_stdout(io.StringIO()):      # silence the [init] print
        ex = rx.MVEExplorer(
            emb_dict=emb_dict, pool_smiles=pool_smiles, oracle=oracle,
            surrogate_type="single", backbone="molformer", acq="greedy",
            init_size=100, batch_size=100, n_rounds=rounds, topk=50,
            run_dir=new_run_dir(), seed=seed, surrogate_epochs=20,
        )
    if make_surrogate is not None:
        ex.model.surrogate = make_surrogate()
    with contextlib.redirect_stdout(io.StringIO()):      # silence the per-round prints
        hist = ex.run()
    return [h["topk_recall"] for h in hist]

print("real SingleBackboneMVESurrogate:", [f"{r:.0%}" for r in run_with()])
print("our LinearEnsembleSurrogate:    ", [f"{r:.0%}" for r in run_with(LinearEnsembleSurrogate)])

Our fake scores are an almost-linear function of the embedding, so a linear model is a
strong choice here. On real docking scores it wouldn't be. Don't read a toy comparison
as a real result. The point is that **the plumbing works**.

---
## Summary

| Idea | Where you met it in al-molecular |
|---|---|
| class, `__init__`, `self`, attributes | every surrogate; state like `_ym`, `_ys`, the network weights |
| interface / duck typing | `fit(X, y)` and `predict(X) -> (mu, sigma)` in `surrogates.py` |
| abstract base class | `molpal.models.base.Model` |
| inheritance, `super()` | `SingleBackboneEmbeddingModel`, `ParallelMVEExplorer`, `AdaptiveMVEExplorer` |
| composition | Explorer `.model` → Model `.surrogate`; `EnsembleFusionSurrogate` holds one surrogate per backbone |
| functions as values, factories | `get_metric()`, `mve()` |
| `**kwargs`, keyword-only `*` | `mve(..., **kwargs)`, `surrogate_kwargs`, `train(xs, ys, *, epochs=...)` |
| `__call__`, `@property` | `DictObjective`, `Model.__call__`, `provides`, `n_components` |
| `hasattr` / `getattr` branches | `combine` (rank fusion) and `needs_smiles` (fine-tuning) |
| `argparse`, `__main__` | `parse_args()`, `main()` |
| `json`, `pickle`, resume | `_checkpoint`, `find_resume_checkpoint`, `--resume` |

**The whole call flow in one line:**
`main` → `parse_args` → `load_oracle` / `load_embeddings` → `MVEExplorer.__init__`
(→ `mve` → surrogate + wrapper) → `run`: [`model.train` → `surrogate.fit`] →
[`_chunked_get_means_and_vars` → `model.get_means_and_vars` → `surrogate.predict`]
→ `acq_fn` → argsort → oracle → `_checkpoint`, repeated each round.

## Exercises

Answers: [../beginner_tour/solutions/5_solutions.ipynb](../beginner_tour/solutions/5_solutions.ipynb). Try first!

1. **Subclass the explorer.** Write `class LoudExplorer(rx.MVEExplorer)` that overrides
   `_checkpoint(self, rnd, record)` to print the 3 best molecules found so far, then
   calls `super()._checkpoint(rnd, record)` so files are still written. Run it for 2 rounds.
2. **A new surrogate.** Write `RandomSurrogate` whose `predict` returns random `mu` and
   zero `sigma`. Swap it in as in Part 11. Its recall should match random search, about
   (labeled molecules ÷ 3000).
3. **Break the contract on purpose.** Make `predict` return only `mu` (not a tuple). Which
   line of the real code fails, and with what error? Use the traceback to find it.
4. **Trace another surrogate.** Run Part 8c with `surrogate_type="ltall"` (and maybe
   `surrogate_kwargs={"epochs": 5}`). Which branch of `_chunked_get_means_and_vars` does
   it take, and why? (Check Part 7's table.)
5. **Read the parallel loop.** Open `ParallelMVEExplorer.run` in `run_experiment.py`. Which
   lines replace the `_chunked_get_means_and_vars` call from 8b? (Hint: look for `marker`
   and `done`.) `06_al_molecular_tour/mini_shard_workers.py` is a runnable miniature.